# RenewableRessourcesJ

> JAX version of the Renewable Ressources environment.

In [ ]:
#| default_exp Environments/RenewableRessourcesJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from pyCRLD.Environments.BaseJ import ebaseJ
from pyCRLD.Utils.HelpersJ import make_variable_vectorJ
from fastcore.utils import *
from fastcore.test import *
from typing import Iterable
import jax
import jax.numpy as jnp
import jax.scipy.stats as jstats
import numpy as np

In [ ]:
#| export
class RenewableRessourcesJ(ebaseJ):
    """
    Environment with Renewable Ressources.
    """
    def __init__(self, r, C, pR=0.1, obs=None, deltaE=0.2, sig=1.0):
        self.r = r    # regrowth rate
        self.C = C    # Capacity
        self.pR = pR  # recovery propbability in the case of depeletion

        self.N = 1  # starting with one agent, but this could be made adaptive
        self.M = 3  # 2 for now, but eventually three?
        self.Z = len(self._growth_dict())

        self.obs = obs

        self.dE = deltaE  # difference from max_sus_yield form low and high
        self.sig = sig  # std of normal for state transitions

        self.T = self.TransitionTensor()
        self.R = self.RewardTensor()
        self.O = self.ObservationTensor()
        super().__init__()

In [ ]:
#| export
@patch
def _growth(self: RenewableRessourcesJ, stock):
    return self.r * stock * (1 - stock / self.C)


@patch
def _growth_dict(self: RenewableRessourcesJ):
    gdic = {0: self._growth(0)}

    stock = 1
    while self._growth(stock) > 0:
        gdic[stock] = self._growth(stock)
        stock += 1

    return gdic


@patch
def _action_values(self: RenewableRessourcesJ):
    """
    What are the extraction levels corresponding to actions?
    TODO: To be adjusted when multi agent system is considered.
    """
    gdic = self._growth_dict()
    max_sus_yield = max(gdic.values())
    zer_extract = 0
    low_extract = (1 - self.dE) * max_sus_yield
    hig_extract = (1 + self.dE) * max_sus_yield
    return zer_extract, low_extract, hig_extract

@patch
def actions(self: RenewableRessourcesJ):
    z, l, h = self._action_values()
    return [0, 1, 2], ["0", "low", "high"]


@patch
def states(self: RenewableRessourcesJ):
    return [i for i in range(len(self._growth_dict()))]

In [ ]:
#| export
@patch
def TransitionTensor(self: RenewableRessourcesJ):
    """Get the Transition Tensor."""
    dim = np.concatenate(([self.Z], [self.M for _ in range(self.N)], [self.Z]))
    Tsas = np.ones(dim) * (-1)

    for index, _ in np.ndenumerate(Tsas):
        Tsas[index] = self._transition_probability(index[0],
                                                   index[1:-1],
                                                   index[-1])
    return jnp.array(Tsas)

In [ ]:
#| export
@patch
def _transition_probability(self: RenewableRessourcesJ, s, jA, sprim):
    acts = np.array(jA)
    act_vals = np.array(self._action_values())

    total_harvest = sum(act_vals[acts])
    harvest_stock = max(s - total_harvest, 0)
    new_stock = max(harvest_stock + self._growth(harvest_stock),
                    self._recoverP(jA))
    new_stock = min(new_stock, self.Z - 1)

    # gaussian distribution with std `sig` around new_stock
    sig = self.sig

    if sprim == 0:  # minimum
        p = float(jstats.norm.cdf(0.5, new_stock, sig))
    elif sprim == self.Z - 1:  # maximum
        p = float(1 - jstats.norm.cdf(self.Z - 1.5, new_stock, sig))
    else:
        p = float(jstats.norm.cdf(sprim + 0.5, new_stock, sig)
                  - jstats.norm.cdf(sprim - 0.5, new_stock, sig))

    return p

@patch

def _recoverP(self: RenewableRessourcesJ, jA):
    '''
    makes random recovery action dependent.
    It must pay of to choose low at degredation
    '''
    hig_recoverP = (1 + self.dE) * self.pR
    low_recoverP = (1 - self.dE) * self.pR
    zer_recoverP = 0

    recover_vals = np.array([hig_recoverP, low_recoverP, zer_recoverP])

    return recover_vals[jA].mean()

In [ ]:
#| export
@patch

def RewardTensor(self: RenewableRessourcesJ):
    """Get the Reward Tensor R[i,s,a1,...,aN,s']."""
    dim = np.concatenate(([self.N],
                          [self.Z],
                          [self.M for _ in range(self.N)],
                          [self.Z]))
    Risas = np.zeros(dim)

    for index, _ in np.ndenumerate(Risas):
        Risas[index] = self._reward(index[0], index[1], index[2:-1],
                                    index[-1])
    return jnp.array(Risas)

@patch

def _reward(self: RenewableRessourcesJ, i, s, jA, sprim):
    act_vals = np.array(self._action_values())
    reward = 0.1 * act_vals[jA[i]] if s == 0 or sprim == 0\
        else act_vals[jA[i]]
    return reward

In [ ]:
#| export
@patch

def ObservationTensor(self: RenewableRessourcesJ):

    if self.obs is None:
        self.obs = [[s] for s in range(self.Z)]
    self.Q = len(self.obs)

    dim = np.concatenate(([self.N],
                  [self.Z],
                  [self.Q]))
    Oiso = np.zeros(dim)

    for o in range(self.Q):
        for s in self.obs[o]:
            Oiso[:, s, o] = 1

    Oiso = Oiso / Oiso.sum(-1, keepdims=True)

    return Oiso

@patch
def id(self: RenewableRessourcesJ):
    """
    Returns id string of environment TODO
    """
    # Default
    def shorten(a):
        return a

    r = shorten(self.r)
    C = shorten(self.C)

    id = f"{self.__class__.__name__}_"+\
        f"{self.N}_{str(r)}_{str(C)}"

    return id

## Tests

In [ ]:
env = RenewableRessourcesJ(r=1.0, C=10)
assert isinstance(env.T, jnp.ndarray)
assert jnp.allclose(env.T.sum(-1), 1.0, atol=1e-5)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()